# DNT Organoid AI — High-Memory Cloud Quantification Bridge
## Reference Index Preparation & Persistent Storage Synchronization

**Study Accession:** BioProject `PRJNA802072` (SRA `SRP357550`)
**Target Runtime:** Google Colab High-RAM Instance (83.5 GB RAM, 12 vCPUs, ~235 GB Local NVMe/SSD)
**Storage Architecture:** Local NVMe/SSD for High-IOPS Active Processing + Google Drive for Persistent Verification Checkpoints
**Reference Build:** Ensembl GRCh38 Release 112 (Primary Assembly + Annotation GTF + Deterministic BED12)

> **GOVERNANCE MANDATE:**
> 1. **Local Scratch for Active Work:** Active STAR indexing, FASTQ streaming, and samtools sorting execute on local NVMe (`/content`) to avoid Google Drive FUSE latency and rate limits.
> 2. **Single-Archive Verified Persistence:** The completed STAR index is archived into a single verified tarball for Google Drive transfer, minimizing Drive API requests and FUSE overhead while preserving file-level integrity.
> 3. **Account Quota Pre-Flight:** Google Drive capacity is programmatically probed against your account's quota (e.g. 5 TB) before transfers to ensure sufficient free space + safety margin.
> 4. **Scientific Review Gate:** Processing strictly pauses after reference-index generation and validation. No FASTQ sequencing libraries are downloaded until review is complete.

### Stage 1: Google Drive Mount & Account Quota Configuration
Mount Google Drive after user authorization. Configure the persistent storage directory and account capacity (5 TB total capacity).

In [ ]:
import os
import sys
from pathlib import Path
import shutil

# Check if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Account-level storage configuration (user quota: 5 TB)
TOTAL_ACCOUNT_CAPACITY_GB = 5000.0

if IN_COLAB:
    from google.colab import drive
    print("Requesting Google Drive mount authorization...")
    drive.mount('/content/drive')
    # Parameterized persistent root (never hardcoded to an individual email/account)
    PERSISTENCE_ROOT = Path('/content/drive/MyDrive/DNT_Organoid_AI_Persistence')
    print(f"[OK] Google Drive mounted. Persistent storage root: {PERSISTENCE_ROOT}")
else:
    PERSISTENCE_ROOT = Path('/tmp/DNT_Organoid_AI_Persistence')
    print(f"Running outside Colab. Fallback persistent storage root: {PERSISTENCE_ROOT}")

PERSISTENCE_ROOT.mkdir(parents=True, exist_ok=True)

# Verify capacity
total, used, free = shutil.disk_usage(PERSISTENCE_ROOT)
print(f"Mounted FUSE Free Space: {round(free / (1024**3), 2)} GB")
print(f"Configured Account Total Quota: {TOTAL_ACCOUNT_CAPACITY_GB} GB (5 TB)")

### Stage 2: Fast Bioinformatics Toolchain Provisioning (Micromamba)
Install the pinned bioinformatics toolchain into Colab runtime:
- **STAR 2.7.11b** (Splice-aware reference aligner)
- **fastp 0.23.4** (Adapter and poly-G trimmer)
- **subread 2.0.6 / featureCounts** (Gene-level summarization)
- **samtools 1.19.2** (BAM sorting and indexing)
- **rseqc 5.0.2 / infer_experiment.py** (Strandedness verification)

In [ ]:
%%bash
# Fast micromamba provisioning in /content/micromamba
if [ ! -f "/content/micromamba/bin/micromamba" ]; then
  echo "Downloading standalone micromamba binary..."
  mkdir -p /content/micromamba
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/micromamba bin/micromamba
fi

if [ ! -d "/content/dnt_env" ]; then
  echo "Creating pinned bioinformatics environment (/content/dnt_env)..."
  /content/micromamba/bin/micromamba create -y -p /content/dnt_env \
    -c conda-forge -c bioconda \
    star=2.7.11b \
    fastp=0.23.4 \
    subread=2.0.6 \
    samtools=1.19.2 \
    rseqc=5.0.2 \
    python=3.12 \
    pyyaml \
    pandas \
    pytest
fi

# Test tools in environment
export PATH="/content/dnt_env/bin:$PATH"
echo "Toolchain Verification:"
STAR --version
fastp --version
featureCounts -v 2>&1 | head -n 2
samtools --version | head -n 1
infer_experiment.py --version 2>&1


### Stage 3: Secure Repository Setup on Local High-Speed SSD
Clone the approved development branch (`feature/fastq-sra-real-data-validation`) to `/content/DNT-Organoid-AI`.
Authentication uses Colab Secrets (`userdata.get('GITHUB_TOKEN')`) or secure non-echoing `getpass`, never hardcoding tokens.

In [ ]:
import os
import sys
import base64
import subprocess
from pathlib import Path

# Prepend conda environment to PATH
os.environ["PATH"] = f"/content/dnt_env/bin:{os.environ.get('PATH', '')}"

REPO_DIR = Path("/content/DNT-Organoid-AI")
BRANCH = "feature/fastq-sra-real-data-validation"
REPO_URL = "https://github.com/bioprintingLabsInc/DNT-Organoid-AI.git"

if not REPO_DIR.exists():
    print(f"Cloning DNT-Organoid-AI repository ({BRANCH}) to local SSD...")
    # Check Colab Secrets for optional GITHUB_TOKEN
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    
    if not token:
        # Attempt public clone first
        res = subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            import getpass
            print("Repository requires authentication. Please enter GitHub Access Token (hidden input):")
            token = getpass.getpass("Token: ")
    
    if token and not REPO_DIR.exists():
        # Authenticated clone using Authorization header (never embeds token in remote URL or .git/config)
        auth_b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        extra_header = f"http.extraHeader=AUTHORIZATION: basic {auth_b64}"
        res = subprocess.run(
            ["git", "-c", extra_header, "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
            capture_output=True,
            text=True,
        )
        if res.returncode != 0:
            clean_err = res.stderr.replace(auth_b64, "[REDACTED]")
            print(f"Clone failed (code {res.returncode}): {clean_err}")
            raise RuntimeError("Failed to clone repository with provided credentials.")
        subprocess.run(["git", "-C", str(REPO_DIR), "config", "http.extraHeader", f"AUTHORIZATION: basic {auth_b64}"], check=True)
        print("[OK] Repository successfully cloned using secure token authentication.")
    elif REPO_DIR.exists():
        print("[OK] Repository successfully cloned via public access.")
else:
    print(f"Repository already present on local disk: {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

%cd /content/DNT-Organoid-AI
!git log -n 1 --oneline


### Stage 4: Hardware Baseline & Execution Profile Preflight
Probe hardware baseline and execute the full synthetic preflight test suite under the `colab_highmem` execution profile.

In [ ]:
from src.quantification_bridge.pipeline import QuantificationBridgePipeline
from src.quantification_bridge.profiles import resolve_pipeline_paths

pipeline = QuantificationBridgePipeline(repo_root=REPO_DIR, profile_name="colab_highmem")
env_metrics = pipeline.record_environment_metrics()

print("=" * 80)
print("HARDWARE BASELINE AUDIT:")
print(f"OS / Kernel:           {env_metrics['uname']}")
print(f"Physical RAM:          {env_metrics['physical_ram_gb']} GB (Requirement: >= 75.0 GB)")
print(f"CPU Cores:             {env_metrics['cpu_cores']} (Requirement: >= 8)")
print(f"Local Scratch Free:    {env_metrics['scratch_disk_free_gb']} GB (Requirement: >= 150.0 GB)")
print("=" * 80)

preflight_res = pipeline.run_profile_checks(enforce_physical_ram=True)
print(f"Preflight Compatibility: {preflight_res.is_compatible}")
print(f"Fatal Errors:            {preflight_res.has_fatal_errors}")
for finding in preflight_res.findings:
    print(f"  [{finding.severity}] {finding.code}: {finding.message}")

assert not preflight_res.has_fatal_errors, "Preflight checks failed! Cannot proceed."

# Run full synthetic preflight test suite
!pytest tests/quantification_bridge/test_preflight_suite.py -v

### Stage 5: Reference Acquisition & Cryptographic Verification
Acquire Ensembl Release 112 GRCh38 primary assembly FASTA and GTF directly from authoritative Ensembl FTP/HTTPS servers. Verify SHA-256 checksums against `reference_manifest.json` before decompression.

In [ ]:
from src.quantification_bridge.checkpoints import CheckpointManager

paths = resolve_pipeline_paths(REPO_DIR)
checkpoint_mgr = CheckpointManager(
    local_checkpoint_dir=paths["data_dir"] / "checkpoints",
    persistent_checkpoint_dir=PERSISTENCE_ROOT / "checkpoints",
)

print("Acquiring and cryptographically verifying reference files from authoritative sources...")
ref_res = pipeline.acquire_and_verify_references()
print("[OK] Reference Verification Complete:")
print(f"  FASTA SHA-256: {ref_res['fasta_sha256']} (VERIFIED MATCH)")
print(f"  GTF SHA-256:   {ref_res['gtf_sha256']} (VERIFIED MATCH)")
print(f"  BED12 SHA-256: {ref_res['bed12_sha256']} (VERIFIED MATCH)")

checkpoint_mgr.record_stage_completed("REFERENCES_VERIFIED", artifacts=[ref_res])

### Stage 6: Monitored STAR Reference-Index Generation on Local High-Speed SSD
Generate the canonical dense STAR genome index (`sparseD 1`, `nbases 14`, `threads 12`, `limitGenomeGenerateRAM 60000000000`).
If a verified index archive already exists in Google Drive, it can be restored directly in seconds.

In [ ]:
import time
from src.quantification_bridge.checkpoints import restore_and_verify_archive

PERSISTENT_REF_DIR = PERSISTENCE_ROOT / "references"
PERSISTENT_ARCHIVE = PERSISTENT_REF_DIR / "star_index_GRCh38_112_dense.tar"
SKIP_IF_ALREADY_ON_DRIVE = True

need_to_build = True
if SKIP_IF_ALREADY_ON_DRIVE and PERSISTENT_ARCHIVE.is_file():
    print(f"[FOUND] Existing verified index archive detected on Google Drive: {PERSISTENT_ARCHIVE.name}")
    print("Restoring and verifying index into local high-speed SSD (/content)... (approx 1-2 minutes)")
    rest_res = restore_and_verify_archive(
        archive_path=PERSISTENT_ARCHIVE,
        target_extract_dir=paths["star_index_dir"],
    )
    print(f"[RESTORED] Restored {rest_res['verified_files_count']} verified files in {rest_res['extract_seconds']}s.")
    need_to_build = False

if need_to_build:
    print("Starting canonical dense STAR genomeGenerate on local SSD (threads=12, limit_ram=60GB)...")
    t0 = time.time()
    idx_res = pipeline.build_star_index(
        threads=12,
        sparse_d=1,  # Canonical dense reference standard on Colab High-RAM
        genome_sa_index_nbases=14,
        limit_ram=60000000000,
    )
    duration_min = (time.time() - t0) / 60
    print(f"[OK] STAR Index generation finished cleanly in {duration_min:.2f} minutes!")

### Stage 7: Index Artifact Inventory & In-Memory Dry-Run Load Test
Verify that all 14 required STAR index binary files exist, are non-empty, and load into memory without processing sequencing reads.

In [ ]:
from scripts.run_colab_reference_indexing import verify_index_artifacts, lightweight_index_load_test

print("Verifying index artifact inventory...")
eval_res = verify_index_artifacts(paths["star_index_dir"])
assert eval_res["is_complete"], f"Index is incomplete! Missing: {eval_res['missing_files']}"

print(f"[OK] All {len(eval_res['file_details'])} required files verified present and valid:")
for fname, detail in eval_res['file_details'].items():
    print(f"  - {fname:<25} {detail['size_mb']:>9.2f} MB")

print(f"\nTotal Local Index Size: {eval_res['total_index_gb']} GB")

# Lightweight dry-run load test
load_test = lightweight_index_load_test(paths["star_index_dir"], star_bin="STAR")
print(f"In-Memory Load Test Status: {load_test['load_test_passed']} (Exit Code: {load_test.get('exit_code')}, Time: {load_test['load_time_seconds']}s)")
assert load_test["load_test_passed"], "In-memory index load test failed!"

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_BUILT_AND_VERIFIED",
    artifacts=[{"dir": str(paths["star_index_dir"]), "total_gb": eval_res["total_index_gb"]}],
)

### Stage 8: Single-Archive Persistent Storage Synchronization
Package the completed STAR index into a single verified tar archive (`star_index_GRCh38_112_dense.tar`) and sync to Google Drive with SHA-256 validation. This avoids Google Drive FUSE latency and rate limits.

In [ ]:
from src.quantification_bridge.checkpoints import archive_and_sync_directory

TARGET_PERSISTENT_REF = PERSISTENCE_ROOT / "references"
SAFETY_MARGIN_GB = 10.0

print("Archiving and synchronizing index to Google Drive...")
archive_res = archive_and_sync_directory(
    src_dir=paths["star_index_dir"],
    dst_dir=TARGET_PERSISTENT_REF,
    archive_name="star_index_GRCh38_112_dense.tar",
    safety_margin_gb=SAFETY_MARGIN_GB,
    account_quota_gb=TOTAL_ACCOUNT_CAPACITY_GB,
    delete_source_after=False,  # Retain local copy on fast SSD for active alignment
)

print("=" * 80)
print(f"[PERSISTENCE VERIFIED] Successfully archived {archive_res['total_files']} files to Google Drive in {archive_res['total_seconds']}s.")
print(f"Destination Archive: {archive_res['dst_archive_path']} ({archive_res['archive_size_gb']} GB)")
print(f"Archive SHA-256:     {archive_res['archive_sha256']}")
print("=" * 80)

checkpoint_mgr.record_stage_completed(
    "STAR_INDEX_PERSISTED_TO_GDRIVE",
    artifacts=[{"archive": archive_res["dst_archive_path"], "sha256": archive_res["archive_sha256"]}],
)

## MANDATORY SCIENTIFIC REVIEW GATE
### EXECUTION HALT

In strict compliance with user governance:
- **Reference Indexing is Complete and Validated.**
- **Persistent Backup is Cryptographically Verified on Google Drive.**
- **DO NOT proceed to FASTQ sequencing library acquisition or alignment until this report is formally reviewed and approved.**
- Run the final cell below to produce the formal readiness decision table.

In [ ]:
print("=" * 80)
print("DNT ORGANOID AI - REFERENCE INDEX VERIFICATION SUMMARY")
print("=" * 80)
print(f"Repository Git Commit:        {os.popen('git rev-parse --short HEAD').read().strip()}")
print(f"Active Profile:               colab_highmem")
print(f"Reference Genome:             Ensembl GRCh38 release 112 (Homo sapiens)")
print(f"Total Local Index Size:       {eval_res['total_index_gb']} GB")
print(f"Archive Size on Drive:        {archive_res['archive_size_gb']} GB (Verified Match)")
print(f"Files Verified:               {len(eval_res['file_details'])}")
print(f"In-Memory Load Status:        PASSED")
print(f"Google Drive Checkpoint:      COMPLETED & PERSISTED")
print(f"Status:                       AWAITING_OWNER_PILOT_FASTQ_AUTHORIZATION")
print("=" * 80)
print("PIPELINE PAUSED: AWAITING USER APPROVAL TO BEGIN 2-SAMPLE TECHNICAL PILOT FASTQ DOWNLOAD.")